In [53]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from collections import Counter
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import os

os.chdir(r"C:\Users\hempe\Studium\Masterthesis\Repository\Masterthesis")

In [54]:
#load data from CSV with unicode encoding
df = pd.read_csv("data/raw/train_data.csv", sep=',', encoding='utf-8')

C:\Users\hempe\AppData\Local\Temp\ipykernel_30132\2260802705.py:2: DtypeWarning: Columns (0: Kd (nM), 1: EC50 (nM), 2: koff (s-1), 3: UniProt (TrEMBL) Submitted Name of Target Chain 2, 4: UniProt (TrEMBL) Entry Name of Target Chain 2, 5: UniProt (TrEMBL) Primary ID of Target Chain 2, 6: UniProt (TrEMBL) Secondary ID(s) of Target Chain 2, 7: BindingDB Target Chain Sequence 3, 8: PDB ID(s) of Target Chain 3, 9: UniProt (SwissProt) Recommended Name of Target Chain 3, 10: UniProt (SwissProt) Entry Name of Target Chain 3, 11: UniProt (SwissProt) Primary ID of Target Chain 3, 12: UniProt (SwissProt) Secondary ID(s) of Target Chain 3, 13: UniProt (TrEMBL) Submitted Name of Target Chain 3, 14: UniProt (TrEMBL) Entry Name of Target Chain 3, 15: UniProt (TrEMBL) Primary ID of Target Chain 3, 16: UniProt (TrEMBL) Secondary ID(s) of Target Chain 3, 17: BindingDB Target Chain Sequence 4, 18: PDB ID(s) of Target Chain 4, 19: UniProt (SwissProt) Recommended Name of Target Chain 4, 20: UniProt (SwissP

In [55]:
# filter all data where IC50 (nM) is not null
df = df[df['IC50 (nM)'].notnull()]
#filter all data where BindingDB Target Chain Sequence 1 is not null
df = df[df['BindingDB Target Chain Sequence 1'].notnull()]
#filter all data where Ligand SMILES is not null
df = df[df['Ligand SMILES'].notnull()]
#filter all data where IC50 (nM) is not numeric
df = df[pd.to_numeric(df['IC50 (nM)'], errors='coerce').notnull()]
#transform column IC50 (nM) to numeric
df["IC50 (nM)"] = pd.to_numeric(df["IC50 (nM)"], errors="coerce")
#filter all rows were IC50 is < 0
df = df[df["IC50 (nM)"] > 0]
#Remove outlier >1e7
df = df[df["IC50 (nM)"] <= 1e7]


In [56]:
#Select relevant feaures
df=df[['Ligand SMILES', 'BindingDB Target Chain Sequence 1', 'IC50 (nM)']]
df.head()

,Ligand SMILES,BindingDB Target Chain Sequence 1,IC50 (nM)
1,CCc1cn2CCS(=O)(=O)Oc3cc(cc1c23)C(=O)N[C@@H](Cc...,MGALARALLLPLLAQWLLRAAPELAPAPFTLPLRVAAATNRVVAPT...,450.0
2,Clc1cccc(Nc2ncnc3n[nH]c(NCc4ccccc4)c23)c1,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...,7.0
3,CC1CCCCCCCC(=O)Cc2c(Cl)c(O)cc(O)c2C(=O)O1,MASETFEFQAEITQLMSLIINTVYSNKEIFLRELISNASDALDKIR...,1290.0
4,COc1cc(ccc1N)-c1ccc2c(n[nH]c2c1)-c1nc2c(cccc2[...,MPSRTGPKMEGSGGRVRLKAHYGGDIFITSVDAATTFEELCEEVRD...,51.3
7,Cc1ccc(cc1)N1CCc2cc(O)ccc2C1(C)c1ccc(OCCN2CCCC...,MDIKNSPSSLNSPSSYNCSQSILPLEHGSIYIPSSYVDSHHEYPAM...,354.0


In [57]:
# Feature Extraktion der SMILES Nomenklatur mittels Morgan Fingerprints
df = df[df["Ligand SMILES"].notna()].copy()

def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return list(AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048))

#Umwandlung in numerischen Vektor
df["fingerprint"] = df["Ligand SMILES"].apply(smiles_to_fp)
df = df[df["fingerprint"].notna()].copy()

In [58]:
#Numerische Repräsentation der Proteinsequenz mittels Amino Acid Composition (AAC) Methode

aa_list = list("ACDEFGHIKLMNPQRSTVWY")

def aac(seq):
    seq = seq.upper()
    counts = Counter(seq)
    length = len(seq)
    return [counts[aa] / length for aa in aa_list]


# Umwandlung der Aminosäuresequenz in einen 20- dimensionalen Vektor
df["AAC"] = df["BindingDB Target Chain Sequence 1"].apply(aac)

In [59]:
df = df[['fingerprint', 'AAC', 'IC50 (nM)']]
df.head()

,fingerprint,AAC,IC50 (nM)
1,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.10038610038610038, 0.015444015444015444, 0....",450.0
2,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.05950413223140496, 0.049586776859504134, 0....",7.0
3,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.06064880112834979, 0.0, 0.06629055007052186...",1290.0
4,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0.04560810810810811, 0.02702702702702703, 0.0...",51.3
7,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...","[0.06415094339622641, 0.03962264150943396, 0.0...",354.0


In [62]:
# Reset index to avoid concat problems
df = df.reset_index(drop=True)

# Fingerprint features
X_fp = pd.DataFrame(df["fingerprint"].tolist())

# AAC features
X_aac = pd.DataFrame(df["AAC"].tolist())

# Combine features
X = pd.concat([X_fp, X_aac], axis=1)

# Number columns
X.columns = range(X.shape[1])

# Target: pIC50
y = -np.log10(df["IC50 (nM)"] * 1e-9)
y.name = "pIC50"

# Combine X and y
df_processed = pd.concat([X, y], axis=1)



In [63]:
# Save train und test data as seperate CSV files
df_processed.to_csv('data/processed/df_train_processed.csv', index=False)
# Print success message
print(f'✅ Df is preprocessed. data size: {df_processed.shape}')

✅ Df is preprocessed. data size: (43705, 2069)
